# HumanEval quality reference for Qwen2.5-Coder-3B

Part of ORBIT-LLM (task SWFW_5S_13). This measures **how much answer quality each weight
precision costs**. It does not measure speed: every timing number in the project comes
from the laptop.

All models run on this one machine with the same settings, so only the weights differ.

**How to use it**
1. Runtime > Change runtime type > **CPU**. OpenVINO cannot use Colab's NVIDIA GPUs.
2. Run all cells in order, then keep this tab open. It may take many hours.
3. If the session stops, open the notebook again and run all cells. It resumes from Drive.
4. When every model is finished or skipped, the last cell downloads the results.
5. Give the downloaded `humaneval_results.zip` to Claude.

In [ ]:
# Models run in this order. FP16 is last because it may not fit in memory.
MODELS = {
    "int4": "OpenVINO/Qwen2.5-Coder-3B-Instruct-int4-ov",
    "int8": "OpenVINO/Qwen2.5-Coder-3B-Instruct-int8-ov",
    "fp16": "OpenVINO/Qwen2.5-Coder-3B-Instruct-fp16-ov",
}
PROBLEMS_URL = "https://github.com/openai/human-eval/raw/master/data/HumanEval.jsonl.gz"
PROBLEMS_SHA256 = "b796127e635a67f93fb35c04f4cb03cf06f38c8072ee7cee8833d7bee06979ef"
DRIVE_FOLDER = "/content/drive/MyDrive/orbit_quality"
RESULTS_ZIP = "humaneval_results.zip"

In [ ]:
!pip -q install "openvino==2026.3.0" "openvino-genai==2026.3.0.0" "openvino-tokenizers==2026.3.0.0"

import platform, shutil, subprocess
import openvino, openvino_genai
print("OpenVINO", openvino.__version__, "| GenAI", openvino_genai.__version__)
print(subprocess.run("lscpu | grep -E 'Model name|^CPU\\(s\\)'; free -g | head -2", shell=True, capture_output=True, text=True).stdout)
print("Free disk:", shutil.disk_usage("/content").free // 1024**3, "GB")

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
Path(DRIVE_FOLDER).mkdir(parents=True, exist_ok=True)
print("Results folder:", DRIVE_FOLDER)

In [ ]:
%%writefile quality_core.py
"""HumanEval scoring: build a test program from a model's answer and run it.

Standard library only. build_notebook.py copies this file into the Colab
notebook, and tests/test_quality_core.py checks it locally.

Model-written code is untrusted. On Colab it runs only after Google Drive has
been unmounted, in a separate process with limits on time, memory and file size.
"""

import json
import math
import os
import re
import signal
import subprocess
import sys
import tempfile
from pathlib import Path

try:
    import resource
except ImportError:  # Windows
    resource = None


INSTRUCTION = (
    "Complete the following Python function. Return the complete function in one "
    "```python code block, including any imports it needs. Do not add tests or "
    "usage examples.\n\n```python\n{prompt}\n```"
)

TIMEOUT_S = 10
MEMORY_LIMIT_BYTES = 2 * 1024**3
FILE_LIMIT_BYTES = 10 * 1024**2

FENCED_BLOCK = re.compile(r"```[a-zA-Z]*[ \t]*\r?\n(.*?)```", re.DOTALL)


def extract_code(text, entry_point):
    """Return the code in a model's answer.

    Prefers the first complete code block that defines the function. An answer
    cut off by the token limit can leave a block unclosed, so that is handled
    too. Text with no code fence is returned unchanged.
    """
    blocks = FENCED_BLOCK.findall(text)
    for block in blocks:
        if f"def {entry_point}" in block:
            return block
    if blocks:
        return blocks[0]
    if "```" in text:
        after_fence = text.split("```", 1)[1]
        return after_fence.split("\n", 1)[1] if "\n" in after_fence else ""
    return text


def build_program(problem, answer):
    """Combine the prompt's preamble, the model's code and the official test."""
    entry_point = problem["entry_point"]
    prompt = problem["prompt"]
    code = extract_code(answer, entry_point)

    if re.search(rf"^\s*def {re.escape(entry_point)}\b", code, re.MULTILINE):
        # A full function: keep what the prompt defines before it (imports and
        # helper functions), and replace the function itself.
        match = re.search(rf"^def {re.escape(entry_point)}\b", prompt, re.MULTILINE)
        preamble = prompt[: match.start()] if match else ""
        body = preamble + code
    else:
        # Only the function body: it continues the prompt.
        body = prompt + code

    return f"{body}\n\n{problem['test']}\n\ncheck({entry_point})\n"


def _limit_resources():
    resource.setrlimit(resource.RLIMIT_AS, (MEMORY_LIMIT_BYTES, MEMORY_LIMIT_BYTES))
    resource.setrlimit(resource.RLIMIT_FSIZE, (FILE_LIMIT_BYTES, FILE_LIMIT_BYTES))
    resource.setrlimit(resource.RLIMIT_CORE, (0, 0))


def run_program(program, timeout_s=TIMEOUT_S):
    """Run a test program. Returns {"outcome": passed|failed|timeout, "detail": ...}."""
    with tempfile.TemporaryDirectory(prefix="humaneval_") as folder:
        (Path(folder) / "candidate.py").write_text(program, encoding="utf-8")
        options = {}
        if resource is not None:
            options = {"preexec_fn": _limit_resources, "start_new_session": True}

        environment = {"PYTHONDONTWRITEBYTECODE": "1", "HOME": folder}
        if os.name == "nt":  # Python cannot start on Windows without this
            environment["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", "")

        process = subprocess.Popen(
            [sys.executable, "-I", "candidate.py"],
            cwd=folder,
            env=environment,
            stdin=subprocess.DEVNULL,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
            text=True,
            encoding="utf-8",
            errors="replace",
            **options,
        )
        try:
            _, errors = process.communicate(timeout=timeout_s)
        except subprocess.TimeoutExpired:
            if resource is not None:
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.communicate()
            return {"outcome": "timeout", "detail": f"over {timeout_s} s"}

    if process.returncode == 0:
        return {"outcome": "passed", "detail": ""}
    lines = errors.strip().splitlines()
    return {"outcome": "failed", "detail": lines[-1][:200] if lines else "no error text"}


def load_jsonl(path):
    """Read a JSON-lines file, skipping a last line cut off by a disconnect."""
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return rows


def end_with_newline(path):
    """Close a line cut off by a disconnect, so the next row starts cleanly."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return
    with path.open("rb+") as file:
        file.seek(-1, os.SEEK_END)
        if file.read(1) != b"\n":
            file.write(b"\n")


def wilson_interval(passed, total, z=1.96):
    """95% confidence interval for a proportion."""
    if total == 0:
        return (0.0, 1.0)
    share = passed / total
    denominator = 1 + z * z / total
    centre = (share + z * z / (2 * total)) / denominator
    margin = z * math.sqrt(share * (1 - share) / total + z * z / (4 * total**2)) / denominator
    return (max(0.0, centre - margin), min(1.0, centre + margin))


def paired_difference_interval(only_first, only_second, total, z=1.96):
    """95% confidence interval for first's pass rate minus second's, same problems."""
    difference = (only_first - only_second) / total
    variance = (only_first + only_second - (only_first - only_second) ** 2 / total) / total**2
    margin = z * math.sqrt(variance)
    return (difference - margin, difference + margin)


def mcnemar_exact(only_first, only_second):
    """Two-sided exact McNemar p-value from the problems the two models split on."""
    discordant = only_first + only_second
    if discordant == 0:
        return 1.0
    smaller = min(only_first, only_second)
    tail = sum(math.comb(discordant, k) for k in range(smaller + 1)) / 2**discordant
    return min(1.0, 2 * tail)

In [ ]:
%%writefile run_generation.py
"""Generate one model's HumanEval answers. Resumes after a disconnect.

Runs as its own process, so a model too large for this machine ends only this
process. Exit code -9 means the system ran out of memory.

    python run_generation.py --model-dir MODEL --tag int4 --problems HumanEval.jsonl.gz --out answers_int4.jsonl
"""

import argparse
import gzip
import json
import os
import platform
import time
from pathlib import Path

import openvino_genai as ov_genai

from quality_core import INSTRUCTION, end_with_newline, load_jsonl


MAX_NEW_TOKENS = 512


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--model-dir", required=True)
    parser.add_argument("--tag", required=True)
    parser.add_argument("--problems", required=True)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--limit", type=int, default=None, help="Stop after this many problems.")
    return parser.parse_args()


def main():
    args = parse_args()
    with gzip.open(args.problems, "rt", encoding="utf-8") as file:
        problems = [json.loads(line) for line in file]
    if args.limit:
        problems = problems[: args.limit]

    done = {row["task_id"] for row in load_jsonl(args.out)}
    remaining = [problem for problem in problems if problem["task_id"] not in done]
    print(f"[{args.tag}] {len(done)} done, {len(remaining)} to do", flush=True)
    if not remaining:
        return

    # Every model runs with the same attention backend and execution precision,
    # so only the weights differ between them.
    started = time.perf_counter()
    pipeline = ov_genai.LLMPipeline(
        args.model_dir,
        "CPU",
        ATTENTION_BACKEND="PA",
        INFERENCE_PRECISION_HINT="f32",
    )
    print(f"[{args.tag}] loaded in {time.perf_counter() - started:.0f} s", flush=True)

    config = pipeline.get_generation_config()
    config.do_sample = False
    config.repetition_penalty = 1.0
    config.max_new_tokens = MAX_NEW_TOKENS

    args.out.parent.mkdir(parents=True, exist_ok=True)
    end_with_newline(args.out)
    with args.out.open("a", encoding="utf-8") as results:
        for number, problem in enumerate(remaining, 1):
            history = ov_genai.ChatHistory()
            history.append(
                {"role": "user", "content": INSTRUCTION.format(prompt=problem["prompt"])}
            )

            started = time.perf_counter()
            output = pipeline.generate(history, config)
            seconds = time.perf_counter() - started
            tokens = output.perf_metrics.get_num_generated_tokens()

            results.write(
                json.dumps(
                    {
                        "task_id": problem["task_id"],
                        "tag": args.tag,
                        "answer": output.texts[0],
                        "tokens": tokens,
                        "seconds": round(seconds, 2),
                        "machine": platform.processor() or platform.machine(),
                    }
                )
                + "\n"
            )
            results.flush()
            os.fsync(results.fileno())  # survive a Colab disconnect

            left_hours = seconds * (len(remaining) - number) / 3600
            print(
                f"[{args.tag}] {number}/{len(remaining)} {problem['task_id']}: "
                f"{tokens} tokens, {tokens / seconds:.1f} tok/s, about {left_hours:.1f} h left",
                flush=True,
            )


if __name__ == "__main__":
    main()

In [ ]:
import hashlib, urllib.request
urllib.request.urlretrieve(PROBLEMS_URL, "HumanEval.jsonl.gz")
digest = hashlib.sha256(Path("HumanEval.jsonl.gz").read_bytes()).hexdigest()
assert digest == PROBLEMS_SHA256, "HumanEval file differs from the one checked on the laptop"
print("HumanEval verified")

In [ ]:
import json, shutil, subprocess, sys
from huggingface_hub import snapshot_download
import quality_core as qc

status = {}
for tag, repository in MODELS.items():
    answers = f"{DRIVE_FOLDER}/answers_{tag}.jsonl"
    model_dir = f"/content/models/{tag}"
    if len(qc.load_jsonl(answers)) >= 164:
        status[tag] = "finished"
        print(f"--- {tag}: already finished", flush=True)
        continue
    print(f"--- {tag}: downloading {repository}", flush=True)
    snapshot_download(repository, local_dir=model_dir)
    # Pipe the output here: a child process's prints do not reach the notebook.
    process = subprocess.Popen(
        [sys.executable, "-u", "run_generation.py", "--model-dir", model_dir, "--tag", tag,
         "--problems", "HumanEval.jsonl.gz", "--out", answers],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    status[tag] = "finished" if code == 0 else ("out of memory" if code == -9 else f"failed ({code})")
    print(f"--- {tag}: {status[tag]}", flush=True)
    shutil.rmtree(model_dir, ignore_errors=True)

Path(f"{DRIVE_FOLDER}/status.json").write_text(json.dumps(status))
print(status)

In [ ]:
# Copy the answers, then unmount Drive BEFORE any model-written code runs.
import gzip, json, shutil, zipfile
from collections import Counter
from google.colab import drive, files
import quality_core as qc

Path("scored").mkdir(exist_ok=True)
for path in Path(DRIVE_FOLDER).glob("answers_*.jsonl"):
    shutil.copy(path, "scored")
drive.flush_and_unmount()
print("Drive unmounted")

with gzip.open("HumanEval.jsonl.gz", "rt", encoding="utf-8") as file:
    problems = {row["task_id"]: row for row in map(json.loads, file)}

# The harness must pass the official solutions before its scores mean anything.
official = Counter(
    qc.run_program(
        f"{p['prompt']}{p['canonical_solution']}\n\n{p['test']}\n\ncheck({p['entry_point']})\n"
    )["outcome"]
    for p in problems.values()
)
print("Official solutions:", dict(official))
assert official["passed"] == len(problems), "Scoring harness is broken: do not trust the results"

scores = {}
for path in sorted(Path("scored").glob("answers_*.jsonl")):
    tag = path.stem.removeprefix("answers_")
    rows = qc.load_jsonl(path)
    outcomes = {}
    with open(f"scored/outcomes_{tag}.jsonl", "w", encoding="utf-8") as file:
        for row in rows:
            result = qc.run_program(qc.build_program(problems[row["task_id"]], row["answer"]))
            outcomes[row["task_id"]] = result["outcome"] == "passed"
            file.write(json.dumps({**row, **result}) + "\n")
    scores[tag] = outcomes
    passed, total = sum(outcomes.values()), len(outcomes)
    low, high = qc.wilson_interval(passed, total)
    print(f"{tag}: {passed}/{total} passed = {passed / total:.1%} (95% CI {low:.1%} to {high:.1%})")

reference = "fp16" if len(scores.get("fp16", {})) == len(problems) else "int8"
for tag, outcomes in scores.items():
    if tag == reference or reference not in scores:
        continue
    shared = outcomes.keys() & scores[reference].keys()
    only_reference = sum(scores[reference][t] and not outcomes[t] for t in shared)
    only_this = sum(outcomes[t] and not scores[reference][t] for t in shared)
    print(f"{reference} vs {tag}: {only_reference} only {reference}, {only_this} only {tag}, "
          f"McNemar p = {qc.mcnemar_exact(only_reference, only_this):.3f}")

with zipfile.ZipFile(RESULTS_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in Path("scored").iterdir():
        archive.write(path, path.name)
files.download(RESULTS_ZIP)